# 01 — Preparação e construção inicial da tabela mestra

**Escopo do notebook:** inspeção das fontes, preparação das identificações e construção de um calendário diário inicial com contagens de alarmes. O alvo, as sequências e o modelo serão desenvolvidos em etapas posteriores. Os arquivos de origem permanecem intactos.

A futura unidade de análise será **estação + plataforma + porta + dia**, verificando se o tipo de ativo também precisa compor a chave. O alvo pretendido é a ocorrência de falha entre D+1 e D+7. ZM01 é evidência principal; ZM02 e ZM03 descrevem histórico de manutenção. Manutenção programada ou preventiva não equivale automaticamente a falha.

O notebook utiliza exclusivamente os quatro arquivos tratados em `data/`. As abas de auditoria são inspecionadas separadamente, sem concatená-las às bases principais.

In [1]:
from pathlib import Path
import re
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)

# Funciona iniciando o Jupyter na raiz ou em uma subpasta do repositório.
candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos
             if (p / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("O notebook precisa ser executado a partir do repositório que contém data/.")

FONTES = {
    "Alarmes": (RAIZ / "data/Alarmes/alarmes_trata_limpos_consolidado.csv", None),
    "ZM01": (RAIZ / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx", "ZM01_limpa"),
    "ZM02": (RAIZ / "data/Corretiva Programada/processados/ZM02_limpa.xlsx", "ZM02_limpa"),
    "ZM03": (RAIZ / "data/Preventiva/preventiva_zm03_limpa.csv", None),
}
for nome, (arquivo, _) in FONTES.items():
    if not arquivo.is_file():
        raise FileNotFoundError(f"{nome}: {arquivo}")
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"Raiz: {RAIZ}")


Python: 3.13.14 | pandas: 2.3.1
Raiz: C:\Users\Inteli\Downloads\EC-M7\g04


## 1. Inventário e leitura

Os CSVs são lidos em UTF-8. As abas são escolhidas explicitamente para não confundir a base limpa com auditorias ou subconjuntos. A contagem de linhas abaixo corresponde aos registros lidos, não à quantidade de episódios de falha.

In [2]:
bases = {}
auditorias = {}
inventario = []
for nome, (arquivo, principal) in FONTES.items():
    if principal is None:
        df = pd.read_csv(arquivo, encoding="utf-8-sig", low_memory=False)
        bases[nome] = df
        inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                           "aba": "(CSV)", "papel": "principal",
                           "linhas": len(df), "colunas": len(df.columns)})
    else:
        with pd.ExcelFile(arquivo) as livro:
            for aba in livro.sheet_names:
                df = pd.read_excel(livro, sheet_name=aba)
                if aba == principal:
                    bases[nome] = df
                else:
                    auditorias[(nome, aba)] = df
                inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                                   "aba": aba, "papel": "principal" if aba == principal else "auditoria",
                                   "linhas": len(df), "colunas": len(df.columns)})
display(pd.DataFrame(inventario))
for (nome, aba), df in auditorias.items():
    if aba == "Decisoes_limpeza":
        print(f"Decisões de limpeza — {nome}")
        display(df)


,fonte,arquivo,aba,papel,linhas,colunas
0,Alarmes,data\Alarmes\alarmes_trata_limpos_consolidado.csv,(CSV),principal,742506,15
1,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,ZM01_limpa,principal,10543,27
2,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Outliers_removidos,auditoria,103,27
3,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
4,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Decisoes_limpeza,auditoria,7,2
5,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,ZM02_limpa,principal,1163,27
6,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_extremas,auditoria,12,27
7,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_zero,auditoria,5,27
8,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
9,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Decisoes_limpeza,auditoria,9,2


Decisões de limpeza — ZM01


,campo_ou_regra,decisao
0,iqm,Coluna removida por conter 100% de valores ausentes.
1,duplicatas,Removidas 1 linha(s) exatamente duplicada(s).
2,duracao_avaria_horas,"Removidos 103 registros acima do percentil 99 (25.00 horas), preservados separadamente para audi..."
3,fim_da_avaria,Ausências preservadas; não foi realizada imputação.
4,data_analise,Ausências preservadas; marca e data de análise não são equivalentes.
5,campos_catalogo,"Ausências de objeto, defeito, causa e ação foram preservadas."
6,ordens_ausentes,Registros sem ordem foram mantidos quando possuíam nota e dados úteis.


Decisões de limpeza — ZM02


,campo_ou_regra,decisao
0,falha,Removida por conter 100% de valores ausentes.
1,iqm,Removida por conter 100% de valores ausentes.
2,duplicatas,Nenhuma duplicata exata encontrada após a limpeza.
3,ordens_repetidas,Cinco ordens repetidas foram mantidas porque apresentam divergências em campos técnicos.
4,ordens_ausentes,"Cinco registros sem ordem foram mantidos por possuírem nota, descrição e localização."
5,valores_ausentes,As ausências foram preservadas sem imputação.
6,duracao_extrema,Foram sinalizados 12 registros acima do percentil 99 (6104.70 horas). Os registros foram mantido...
7,duracao_zero,"Foram sinalizados 5 registros com duração igual a zero, mantidos sem imputação."
8,colunas_esparsas,As colunas comentarios_classificacao_engenharia e analise_datetime foram removidas da base final...


## 2. Colunas, tipos e ausências

Os tipos são inferidos na leitura. Esta inspeção preserva os valores ausentes e todos os registros. As amostras exibem somente identificação e datas, evitando reproduzir textos longos das ordens.

In [3]:
AMOSTRAS = {
    "Alarmes": ["TAG", "estacao_extraida", "plataforma_extraida", "porta_extraida",
                "tipo_evento_extraido", "dt_alarme", "dt_normalizacao"],
    "ZM01": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM02": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM03": ["Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada"],
}
for nome, df in bases.items():
    print(f"\n{nome}")
    perfil = pd.DataFrame({
        "coluna": df.columns,
        "tipo_lido": df.dtypes.astype(str).values,
        "ausentes": df.isna().sum().values,
        "ausentes_pct": df.isna().mean().mul(100).round(2).values,
        "strings_vazias": [df[c].map(lambda v: isinstance(v, str) and not v.strip()).sum()
                           if df[c].dtype == "object" else 0 for c in df.columns],
    })
    display(perfil)
    display(df[AMOSTRAS[nome]].head(5))



Alarmes


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,TAG,object,0,0.00,0
1,Horário do Alarme,object,0,0.00,0
2,Estação,object,0,0.00,0
3,Porta,float64,170,0.02,0
4,Alarme,object,0,0.00,0
5,Horário da Normalizacao,object,0,0.00,0
6,Tempo de Normalização,object,0,0.00,0
7,tempo_normalizacao_seg,int64,0,0.00,0
8,dt_alarme,object,0,0.00,0
9,dt_normalizacao,object,0,0.00,0


,TAG,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido,dt_alarme,dt_normalizacao
0,PSD_FAL_FAL2_PDM09_OBS,FAL,FAL2,9.0,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670
1,PSD_LUZ_LUZ1_PDM08_OBS,LUZ,LUZ1,8.0,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917
2,PSD_FAL_FAL1_PDM15_OBS,FAL,FAL1,15.0,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670
3,PSD_LUZ_LUZ1_PDM19_OBS,LUZ,LUZ1,19.0,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917
4,PSD_FRE_FRE1_PDM14_OBS,FRE,FRE1,14.0,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847



ZM01


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,1,0.01,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],326,3.09,0


...

,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,5,0.43,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],39,3.35,0


,nota,ordem,local_de_instalacao,inicio_avaria_datetime,fim_avaria_datetime
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56



ZM03


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,Ordem,int64,0,0.00,0
1,Tipo de ordem,object,0,0.00,0
2,Centro de trabalho,object,0,0.00,0
3,Local de instalação,object,0,0.00,0
4,Texto breve,object,0,0.00,0
5,Plano de manutenção,int64,0,0.00,0
6,Status usuário,object,0,0.00,0
7,Status do sistema,object,0,0.00,0
8,Data-base do início,object,0,0.00,0
9,Hr.início programada,object,0,0.00,0


,Ordem,Local de instalação,DT_Inicio,DT_Fim,cancelada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False


## 3. Datas e período dos registros

A conversão ocorre em séries auxiliares; valores inválidos são contabilizados. Mínimo e máximo são **datas de eventos registrados**, não confirmação de cobertura contínua. Meses sem registros também não demonstram ausência de falhas.

Em ZM01/ZM02, `inicio_avaria_datetime` é um campo candidato a início da avaria, não prova de execução da manutenção ou de quando a informação ficou disponível. Em ZM03, a inspeção compara `DT_Inicio`/`DT_Fim` com os campos de programação: equivalência não comprova execução.

In [4]:
DATAS = {
    "Alarmes": ["dt_alarme", "dt_normalizacao"],
    "ZM01": ["inicio_avaria_datetime", "fim_avaria_datetime", "analise_datetime", "encerramento_datetime"],
    "ZM02": ["inicio_avaria_datetime", "fim_avaria_datetime", "encerramento_datetime"],
    "ZM03": ["DT_Inicio", "DT_Fim", "Data-base do início", "Data-base do fim"],
}
datas = {}
resumo_datas = []
for nome, colunas in DATAS.items():
    for coluna in colunas:
        original = bases[nome][coluna]
        convertida = pd.to_datetime(original, errors="coerce", format="mixed")
        datas[nome, coluna] = convertida
        resumo_datas.append({
            "fonte": nome, "coluna": coluna, "menor_data": convertida.min(),
            "maior_data": convertida.max(), "ausentes_origem": original.isna().sum(),
            "nao_convertidas": (original.notna() & convertida.isna()).sum(),
        })
display(pd.DataFrame(resumo_datas))

PARES = {"Alarmes": ("dt_alarme", "dt_normalizacao"),
         "ZM01": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM02": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM03": ("DT_Inicio", "DT_Fim")}
for nome, (inicio, fim) in PARES.items():
    a, b = datas[nome, inicio], datas[nome, fim]
    print(f"{nome}: fim anterior ao início = {(b < a).sum()}; fim ausente/inválido = {b.isna().sum()}")

mensais = {}
for nome, coluna in {"Alarmes": "dt_alarme", "ZM01": "inicio_avaria_datetime",
                     "ZM02": "inicio_avaria_datetime", "ZM03": "DT_Inicio"}.items():
    serie = datas[nome, coluna].dropna()
    if not serie.empty:
        contagem = serie.dt.to_period("M").value_counts().sort_index()
        mensais[nome] = contagem.reindex(pd.period_range(contagem.index.min(), contagem.index.max(), freq="M"), fill_value=0)
print("Registros por mês (ausente fora do intervalo de eventos de cada fonte):")
display(pd.DataFrame(mensais).sort_index().astype("Int64"))

for derivada, base in [("DT_Inicio", "Data-base do início"), ("DT_Fim", "Data-base do fim")]:
    a, b = datas["ZM03", derivada], datas["ZM03", base]
    validos = a.notna() & b.notna()
    print(f"ZM03: {derivada} e {base}: {validos.sum()} pares válidos; "
          f"{(a[validos].dt.normalize() == b[validos].dt.normalize()).sum()} no mesmo dia.")
display(bases["ZM03"][["Data-base do início", "Hr.início programada", "DT_Inicio",
                       "Data-base do fim", "Fim-base (hora)", "DT_Fim"]].head())


,fonte,coluna,menor_data,maior_data,ausentes_origem,nao_convertidas
0,Alarmes,dt_alarme,2023-02-27 06:26:56.897,2026-05-12 08:16:52.657,0,0
1,Alarmes,dt_normalizacao,2023-02-27 06:27:11.897,2026-05-12 08:16:53.670,0,0
2,ZM01,inicio_avaria_datetime,2023-01-01 02:39:00.000,2026-04-01 22:43:00.000,0,0
3,ZM01,fim_avaria_datetime,2023-01-01 03:39:00.000,2026-04-01 23:25:57.000,326,0
4,ZM01,analise_datetime,2024-11-14 09:59:15.000,2026-05-06 13:39:20.000,5756,0
5,ZM01,encerramento_datetime,2023-01-01 05:16:11.000,2026-04-30 14:01:20.000,1,0
6,ZM02,inicio_avaria_datetime,2023-01-03 03:55:59.000,2026-03-28 14:15:00.000,0,0
7,ZM02,fim_avaria_datetime,2023-01-07 11:30:00.000,2026-05-14 02:00:00.000,39,0
8,ZM02,encerramento_datetime,2023-01-07 17:07:19.000,2026-05-14 04:33:52.000,7,0
9,ZM03,DT_Inicio,2022-12-26 05:30:00.000,2026-03-30 21:30:00.000,0,0


Alarmes: fim anterior ao início = 0; fim ausente/inválido = 0
ZM01: fim anterior ao início = 0; fim ausente/inválido = 326
ZM02: fim anterior ao início = 0; fim ausente/inválido = 39
ZM03: fim anterior ao início = 0; fim ausente/inválido = 0
Registros por mês (ausente fora do intervalo de eventos de cada fonte):


,Alarmes,ZM01,ZM02,ZM03
2022-12,<NA>,<NA>,<NA>,1
2023-01,<NA>,147,51,22
2023-02,50,171,29,16
2023-03,43,178,40,16
2023-04,0,282,31,19
2023-05,0,293,27,18
2023-06,0,292,35,15
2023-07,0,249,18,19
2023-08,0,310,7,12
2023-09,0,199,37,17


ZM03: DT_Inicio e Data-base do início: 569 pares válidos; 569 no mesmo dia.
ZM03: DT_Fim e Data-base do fim: 569 pares válidos; 567 no mesmo dia.


,Data-base do início,Hr.início programada,DT_Inicio,Data-base do fim,Fim-base (hora),DT_Fim
0,2026-03-23,21:30:00,2026-03-23 21:30:00,2026-03-24,04:30:00,2026-03-24 04:30:00
1,2026-03-24,21:30:00,2026-03-24 21:30:00,2026-03-25,04:30:00,2026-03-25 04:30:00
2,2026-03-17,21:30:00,2026-03-17 21:30:00,2026-03-18,04:30:00,2026-03-18 04:30:00
3,2026-03-16,21:30:00,2026-03-16 21:30:00,2026-03-17,04:30:00,2026-03-17 04:30:00
4,2026-03-30,21:30:00,2026-03-30 21:30:00,2026-04-02,04:30:00,2026-04-02 04:30:00


## 4. Identificação dos ativos — extração exploratória

As regras abaixo são deliberadamente conservadoras e criam tabelas auxiliares. Formatos desconhecidos ficam para revisão, sem inferência por texto livre.

- SAP: `SP04-6-PSD-BUT-PLT1-PP05PDM` → BUT / 1 / 05 / PDM.
- Alarme: `PSD_BUT_BUT1_PDM05_OBS` → BUT / 1 / 05 / PDM.
- Registros apenas de estação ou plataforma permanecem nesse nível.
- PDM, PFX e outros tipos não são unificados automaticamente.

A chave completa é provisória e inclui o tipo. Esta etapa mantém os eventos separados e preserva o nível de identificação das manutenções coletivas.

In [5]:
def extrair_ativos(serie, alarme=False):
    texto = serie.astype("string").str.strip().str.upper()
    if alarme:
        padrao = r"^PSD_(?P<estacao>[A-Z]{3})_(?P<estacao_plataforma>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_(?P<evento>[A-Z0-9]+)$"
        out = texto.str.extract(padrao)
        # Aceita PSD_P_BUT1_PDM07_FC e PSD_P_MBI1_PDM10FC.
        alternativo = texto.str.extract(
            r"^PSD_P_(?P<estacao>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_?(?P<evento>[A-Z][A-Z0-9]*)$")
        alternativo["estacao_plataforma"] = alternativo["estacao"]
        out = out.combine_first(alternativo)
        # Tags com estações divergentes não recebem chave válida.
        divergente = out.estacao.ne(out.estacao_plataforma).fillna(False)
        out.loc[divergente, ["estacao", "plataforma", "porta", "tipo_ativo"]] = pd.NA
    else:
        out = texto.str.extract(
            r"^SP\d+-\d+-PSD-(?P<estacao>[A-Z]{3})(?:-|$)")
        out["plataforma"] = texto.str.extract(r"-PLT(\d+)(?:-|$)", expand=False)
        porta = texto.str.extract(r"-PP(?P<porta>\d+)(?P<tipo_ativo>[A-Z]+)$")
        out = out.join(porta)
    out = out[["estacao", "plataforma", "porta", "tipo_ativo"]].copy()
    for coluna in ["plataforma", "porta"]:
        out[coluna] = pd.to_numeric(out[coluna], errors="coerce").astype("Int64").astype("string")
    completa = out.notna().all(axis=1)
    # Nível identificável não significa que um subativo é uma porta.
    out["nivel_identificavel"] = "não reconhecido"
    out.loc[out.estacao.notna(), "nivel_identificavel"] = "estação"
    out.loc[out.estacao.notna() & out.plataforma.notna(), "nivel_identificavel"] = "plataforma"
    out.loc[completa, "nivel_identificavel"] = "porta e tipo"
    out["chave_candidata"] = out.estacao + "|" + out.plataforma + "|" + out.porta + "|" + out.tipo_ativo
    out["identificacao_original"] = serie
    return out

COLUNA_ATIVO = {"Alarmes": "TAG", "ZM01": "local_de_instalacao",
                "ZM02": "local_de_instalacao", "ZM03": "Local de instalação"}
ativos = {nome: extrair_ativos(df[COLUNA_ATIVO[nome]], alarme=nome == "Alarmes")
          for nome, df in bases.items()}
for nome, df in ativos.items():
    print(f"\n{nome} — quantidade de registros por nível identificável")
    display(df.nivel_identificavel.value_counts().rename_axis("nível").to_frame("registros"))
    print("Exemplos de identificação (incluindo formatos que exigem revisão):")
    display(df.drop_duplicates("identificacao_original").groupby("nivel_identificavel", sort=False).head(4))
    print("Tipos identificados:")
    display(df.tipo_ativo.value_counts(dropna=False).to_frame("registros"))



Alarmes — quantidade de registros por nível identificável


,registros
nível,
porta e tipo,742336
não reconhecido,170


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,PSD_FAL_FAL2_PDM09_OBS
1,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,PSD_LUZ_LUZ1_PDM08_OBS
2,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,PSD_FAL_FAL1_PDM15_OBS
3,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,PSD_LUZ_LUZ1_PDM19_OBS
104431,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_LUZ1_CBTC_BIM_FAULT_Value


Tipos identificados:


...

,registros
nível,
plataforma,313
estação,173
não reconhecido,83


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,PTA,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PTA
1,FAL,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-FAL
2,PIN,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PIN
3,BUT,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-BUT
4,VSO,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-VSO-PLT1
9,MBI,2,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT2
10,MBI,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT1
11,PTA,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-PTA-PLT1
260,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,SP04-6-PSD


Tipos identificados:


,registros
tipo_ativo,
<NA>,569


## 5. Compatibilidade das chaves

A interseção abaixo mede apenas a existência da mesma chave nas fontes, **sem associação temporal ou causal**. Uma chave ausente pode decorrer de cobertura, nomenclatura ou falta de eventos. A TAG é comparada aos campos já extraídos dos alarmes para conferir consistência.

In [6]:
a = bases["Alarmes"]
extraido = ativos["Alarmes"]
comparacoes = {
    "estacao": a["estacao_extraida"].astype("string").str.upper().str.strip(),
    "plataforma": a["plataforma_extraida"].astype("string").str.extract(r"(\d+)$", expand=False),
    "porta": pd.to_numeric(a["porta_extraida"], errors="coerce").astype("Int64").astype("string"),
}
for campo, existente in comparacoes.items():
    if campo == "plataforma":
        existente = pd.to_numeric(existente, errors="coerce").astype("Int64").astype("string")
    validos = extraido[campo].notna() & existente.notna()
    print(f"Alarmes — {campo}: {validos.sum()} comparáveis, "
          f"{(extraido.loc[validos, campo] != existente[validos]).sum()} divergências.")

chaves_alarmes = set(extraido.chave_candidata.dropna())
resumo_chaves = []
for nome, df in ativos.items():
    chaves = set(df.chave_candidata.dropna())
    resumo_chaves.append({"fonte": nome, "chaves_completas_distintas": len(chaves),
                          "presentes_nos_alarmes": len(chaves & chaves_alarmes),
                          "ausentes_nos_alarmes": len(chaves - chaves_alarmes)})
display(pd.DataFrame(resumo_chaves))

todos = pd.concat([df[["estacao", "plataforma", "porta", "tipo_ativo"]].dropna().drop_duplicates()
                   for df in ativos.values()]).drop_duplicates()
colisoes = todos.groupby(["estacao", "plataforma", "porta"]).tipo_ativo.nunique()
print("Combinações estação/plataforma/número com mais de um tipo:", colisoes.gt(1).sum())
display(colisoes[colisoes.gt(1)].rename("quantidade_tipos").head(10).to_frame())


Alarmes — estacao: 742336 comparáveis, 0 divergências.


Alarmes — plataforma: 742336 comparáveis, 0 divergências.


Alarmes — porta: 742336 comparáveis, 0 divergências.


,fonte,chaves_completas_distintas,presentes_nos_alarmes,ausentes_nos_alarmes
0,Alarmes,978,978,0
1,ZM01,589,522,67
2,ZM02,518,345,173
3,ZM03,0,0,0


Combinações estação/plataforma/número com mais de um tipo: 475


quantidade_tipos
estacao plataforma porta                  
BUT     1          1                     2
                   10                    2
                   11                    2
                   12                    2
                   13                    2
                   15                    3
                   16                    2
                   17                    2
                   18                    3
                   19                    2

## 6. Duplicidades e flags

Repetição de ordem ou nota não implica automaticamente duplicata: pode representar atualização ou vários registros legítimos. A contagem inclui todas as linhas envolvidas (`keep=False`) e também o excedente de duplicatas exatas. Todos os registros são preservados.

In [7]:
resumo_duplicatas = []
for nome, df in bases.items():
    resumo_duplicatas.append({"fonte": nome,
                              "linhas_em_duplicatas_exatas": df.duplicated(keep=False).sum(),
                              "excedente_duplicatas_exatas": df.duplicated().sum()})
display(pd.DataFrame(resumo_duplicatas))
for nome, colunas in {"ZM01": ["ordem", "nota"], "ZM02": ["ordem", "nota"], "ZM03": ["Ordem"]}.items():
    for coluna in colunas:
        serie = bases[nome][coluna].dropna()
        print(f"{nome}/{coluna}: {serie.duplicated(keep=False).sum()} linhas com identificador repetido; "
              f"{serie[serie.duplicated(keep=False)].nunique()} identificadores repetidos.")

chave_evento = ["TAG", "dt_alarme"]
validos = a[chave_evento].notna().all(axis=1)
print("Alarmes: linhas com TAG + dt_alarme repetidos:",
      a.loc[validos].duplicated(chave_evento, keep=False).sum())
for nome, df in bases.items():
    flags = [c for c in df.columns if c.endswith("_flag") or c == "cancelada"]
    for coluna in flags:
        print(f"{nome} — {coluna}")
        display(df[coluna].value_counts(dropna=False).rename_axis("valor").to_frame("registros"))

# Apenas diagnóstico: não reinserir registros excluídos automaticamente.
excluidos = auditorias.get(("ZM01", "Outliers_removidos"))
if excluidos is not None:
    print(f"ZM01: {len(excluidos)} registros preservados em Outliers_removidos.")
    print("Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.")


,fonte,linhas_em_duplicatas_exatas,excedente_duplicatas_exatas
0,Alarmes,0,0
1,ZM01,0,0
2,ZM02,0,0
3,ZM03,0,0


ZM01/ordem: 136 linhas com identificador repetido; 66 identificadores repetidos.
ZM01/nota: 213 linhas com identificador repetido; 104 identificadores repetidos.
ZM02/ordem: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM02/nota: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM03/Ordem: 0 linhas com identificador repetido; 0 identificadores repetidos.


Alarmes: linhas com TAG + dt_alarme repetidos: 449829
ZM01 — indisponivel_flag


,registros
valor,
True,6972
False,3571


ZM01 — falha_flag


...

ZM02 — duracao_zero_flag


,registros
valor,
False,1158
True,5


ZM03 — cancelada


,registros
valor,
False,559
True,10


ZM01: 103 registros preservados em Outliers_removidos.
Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.


## 7. Decisões necessárias antes da tabela mestra

1. **Ativos:** validar os formatos não reconhecidos, os tipos elegíveis (por exemplo, PDM) e se tipo deve integrar a chave. Não misturar portas homônimas de plataformas diferentes.
2. **Manutenções coletivas:** definir como históricos de estação/plataforma entram como contexto, sem convertê-los em intervenções individuais comprovadas.
3. **Semântica temporal:** confirmar a data efetiva de cada manutenção e quando o registro ficou disponível. Em especial, não tratar datas programadas da ZM03 como prova de execução. Status atuais e textos atualizados depois do evento não reconstituem automaticamente o estado histórico.
4. **Falha:** definir critérios para episódios ZM01 e eventual evidência de defeitos em ZM02, revisar os registros excluídos na limpeza ZM01 e depois estabelecer a regra de associação com alarmes. ZM03 não é alvo positivo por si só.
5. **Cobertura:** confirmar os intervalos completos de cada fonte e do cadastro de ativos; não deduzir início de operação da primeira ocorrência nem cobertura pelo último evento.
6. **Duplicidades:** revisar ordens/notas repetidas antes de contar intervenções ou episódios.

**Resultado da inspeção:** inventário, perfis de qualidade, períodos de eventos e chaves candidatas em memória. As seções seguintes apresentam a preparação das tabelas e o calendário inicial. A definição do alvo permanece pendente.

## 8. Preparar as quatro tabelas — passo a passo

Esta seção prepara os dados para a tabela mestra com operações simples do pandas, sem novas funções.

A preparação reutiliza as identificações verificadas na inspeção (`ativos`). A chave candidata tem o formato `BUT|1|5|PDM`: estação BUT, plataforma 1, número 5, tipo PDM. O tipo permanece na chave para distinguir, por exemplo, PDM e MDU enquanto a relação entre eles não for confirmada.

- `.copy()`: cria uma cópia para trabalhar sem modificar a tabela lida.
- `.rename()`: troca nomes de colunas.
- `.join()`: acrescenta as identificações da inspeção à mesma linha de origem. **Não é um cruzamento entre alarmes e manutenções.**

As linhas sem chave completa continuam nas tabelas. Esta etapa preserva os registros e as datas ausentes; a definição do alvo permanece pendente.

In [8]:
# Colunas de identificação verificadas na inspeção.
colunas_identificacao = [
    "estacao", "plataforma", "porta", "tipo_ativo",
    "nivel_identificavel", "chave_candidata",
]


### 8.1 Alarmes

A seleção inclui a TAG, o tipo do evento e seus horários. `data_alarme` representa o dia da ocorrência, usado mais adiante para contar alarmes.

A normalização pode acontecer depois do dia do alarme. Sua duração final não pode ser usada numa previsão feita antes de o alarme normalizar.

In [9]:
alarmes = bases["Alarmes"][[
    "TAG", "tipo_evento_extraido", "dt_alarme", "dt_normalizacao",
]].copy()

alarmes = alarmes.rename(columns={
    "TAG": "tag",
    "tipo_evento_extraido": "tipo_evento",
    "dt_alarme": "inicio_alarme",
    "dt_normalizacao": "fim_alarme",
})

alarmes = alarmes.join(ativos["Alarmes"][colunas_identificacao])

alarmes["inicio_alarme"] = pd.to_datetime(alarmes["inicio_alarme"], errors="coerce")
alarmes["fim_alarme"] = pd.to_datetime(alarmes["fim_alarme"], errors="coerce")
alarmes["data_alarme"] = alarmes["inicio_alarme"].dt.normalize()

display(alarmes.head())


,tag,tipo_evento,inicio_alarme,fim_alarme,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_alarme
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,2026-05-12
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,2026-05-12
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,2026-05-12
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,2026-05-12
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847,FRE,1,14,PDM,porta e tipo,FRE|1|14|PDM,2026-05-12


### 8.2 ZM01 — corretiva emergencial

A tabela mantém nota, ordem, local e início/fim da avaria. `data_avaria` é uma candidata à data de falha, mas os critérios do alvo e os 103 registros separados na limpeza ainda precisam de revisão. `falha_flag` é preservada para essa revisão; não será usada automaticamente como informação histórica disponível no dia.

In [10]:
zm01 = bases["ZM01"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime", "falha_flag",
]].copy()

zm01 = zm01.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm01 = zm01.join(ativos["ZM01"][colunas_identificacao])

zm01["inicio_avaria"] = pd.to_datetime(zm01["inicio_avaria"], errors="coerce")
zm01["fim_avaria"] = pd.to_datetime(zm01["fim_avaria"], errors="coerce")
zm01["data_avaria"] = zm01["inicio_avaria"].dt.normalize()

display(zm01.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,falha_flag,...,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,14943931,53954586.0,SP04-6-PSD-BUT,2025-11-14 06:35:28,NaT,True,...,<NA>,<NA>,<NA>,estação,<NA>,2025-11-14
1,15133619,54145839.0,SP04-6-PSD-BUT,2026-02-27 07:33:00,2026-02-27 12:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2026-02-27
2,13894133,52913343.0,SP04-6-PSD-BUT-P1S1,2024-04-18 06:00:00,2024-04-18 07:30:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2024-04-18
3,14713349,53724958.0,SP04-6-PSD-BUT-P1S1,2025-07-07 18:38:00,2025-07-08 03:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-07-07
4,14643768,53655728.0,SP04-6-PSD-BUT-P1S1-PSSE1,2025-05-28 09:10:00,2025-05-28 10:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-05-28


### 8.3 ZM02 — corretiva programada

A tabela preserva o nome `inicio_avaria`, conforme a identificação do campo na fonte. Sua interpretação como data de execução da manutenção exige confirmação. Um registro ZM02 não é classificado automaticamente como falha.

In [11]:
zm02 = bases["ZM02"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime",
]].copy()

zm02 = zm02.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm02 = zm02.join(ativos["ZM02"][colunas_identificacao])

zm02["inicio_avaria"] = pd.to_datetime(zm02["inicio_avaria"], errors="coerce")
zm02["fim_avaria"] = pd.to_datetime(zm02["fim_avaria"], errors="coerce")
zm02["data_avaria"] = zm02["inicio_avaria"].dt.normalize()

display(zm02.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-01-25
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-05-30
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-06-19
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-09-30
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56,BUT,1,<NA>,<NA>,plataforma,<NA>,2023-07-21


### 8.4 ZM03 — preventiva

Esta preparação trata `DT_Inicio` e `DT_Fim` como referências de programação, sujeitas à confirmação. O status `cancelada` é mantido para revisão: uma ordem não cancelada não comprova execução.

A inspeção não encontrou identificação individual completa de porta nessa base. Por isso, a preparação mantém as preventivas no nível de identificação disponível, sem atribuí-las a cada porta. Os registros sem estação reconhecida também permanecem disponíveis para revisão.

In [12]:
zm03 = bases["ZM03"][[
    "Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada",
]].copy()

zm03 = zm03.rename(columns={
    "Ordem": "ordem",
    "Local de instalação": "local_original",
    "DT_Inicio": "inicio_programado",
    "DT_Fim": "fim_programado",
})

zm03 = zm03.join(ativos["ZM03"][colunas_identificacao])

zm03["inicio_programado"] = pd.to_datetime(zm03["inicio_programado"], errors="coerce")
zm03["fim_programado"] = pd.to_datetime(zm03["fim_programado"], errors="coerce")
zm03["data_programada"] = zm03["inicio_programado"].dt.normalize()

display(zm03.head())


,ordem,local_original,inicio_programado,fim_programado,cancelada,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_programada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False,PTA,<NA>,<NA>,<NA>,estação,<NA>,2026-03-23
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False,FAL,<NA>,<NA>,<NA>,estação,<NA>,2026-03-24
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False,PIN,<NA>,<NA>,<NA>,estação,<NA>,2026-03-17
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-03-16
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False,VSO,1,<NA>,<NA>,plataforma,<NA>,2026-03-30


## 9. Conferência das identificações

Este resumo conta **registros**, não portas distintas ou falhas. A identificação completa é candidata: ainda depende da escolha dos tipos de ativo que entrarão no modelo.

O pequeno `for` repete a mesma conferência nas quatro tabelas. Os `assert` verificam se todas as linhas e suas posições foram preservadas.

In [13]:
tabelas_preparadas = {"Alarmes": alarmes, "ZM01": zm01, "ZM02": zm02, "ZM03": zm03}
resumo = []

for nome, tabela in tabelas_preparadas.items():
    assert len(tabela) == len(bases[nome]), f"A quantidade de linhas mudou em {nome}."
    assert tabela.index.equals(bases[nome].index), f"A ordem das linhas mudou em {nome}."

    resumo.append({
        "fonte": nome,
        "total_registros": len(tabela),
        "com_chave_completa": tabela["chave_candidata"].notna().sum(),
        "sem_chave_completa": tabela["chave_candidata"].isna().sum(),
    })

display(pd.DataFrame(resumo))


,fonte,total_registros,com_chave_completa,sem_chave_completa
0,Alarmes,742506,742336,170
1,ZM01,10543,9357,1186
2,ZM02,1163,1010,153
3,ZM03,569,0,569


## 10. Primeira contagem diária: alarmes

Esta etapa conta os **registros de alarmes por chave candidata e dia**, com `groupby()` e `size()`.

Por enquanto, a contagem mantém registros repetidos da fonte. Ela é exploratória: não representa episódios únicos e ainda não é a variável final `qtd_alarmes_1d`. As duplicidades e os tipos de ativo precisam de revisão antes da modelagem.

Linhas sem chave ou data ficam em `alarmes_para_revisar`. Não serão atribuídas a uma porta por aproximação. A tabela diária contém somente dias com registros; dias ausentes permanecem sem preenchimento enquanto a cobertura não for confirmada.

In [14]:
# A contagem por ativo exige identificação completa e data.
pode_agrupar = alarmes["chave_candidata"].notna() & alarmes["data_alarme"].notna()

alarmes_identificados = alarmes.loc[pode_agrupar].copy()
alarmes_para_revisar = alarmes.loc[~pode_agrupar].copy()

alarmes_diarios = (
    alarmes_identificados
    .groupby(["chave_candidata", "data_alarme"])
    .size()
    .reset_index(name="qtd_registros_alarmes")
)

# A conferência verifica se houve perda ou contagem duplicada no agrupamento.
assert len(alarmes_identificados) + len(alarmes_para_revisar) == len(alarmes)
assert alarmes_diarios["qtd_registros_alarmes"].sum() == len(alarmes_identificados)
assert not alarmes_diarios.duplicated(["chave_candidata", "data_alarme"]).any()

print("Registros usados na contagem:", len(alarmes_identificados))
print("Registros separados para revisão:", len(alarmes_para_revisar))
display(alarmes_diarios.head(10))


Registros usados na contagem: 742336
Registros separados para revisão: 170


,chave_candidata,data_alarme,qtd_registros_alarmes
0,BUT|1|10|MDU,2023-11-12,1
1,BUT|1|10|MDU,2023-11-13,1
2,BUT|1|10|MDU,2023-11-27,2
3,BUT|1|10|MDU,2023-12-06,11
4,BUT|1|10|MDU,2023-12-31,5
5,BUT|1|10|MDU,2024-01-01,3
6,BUT|1|10|MDU,2024-01-03,4
7,BUT|1|10|MDU,2024-01-08,19
8,BUT|1|10|MDU,2024-01-14,31
9,BUT|1|10|MDU,2024-01-20,5


### Resultado desta etapa

A preparação produz quatro tabelas em memória (`alarmes`, `zm01`, `zm02`, `zm03`) e uma primeira contagem diária (`alarmes_diarios`). Os arquivos de origem continuam intactos e nenhum CSV foi exportado.

**Continuação:** as próximas seções adotam um tipo de ativo e um recorte exploratório para construir o calendário diário e juntar as contagens de alarmes. A confirmação de cobertura, o alvo `falha_7d`, as janelas móveis e o treinamento permanecem para etapas posteriores.

## 11. Tipo de ativo e período inicial

O primeiro calendário considera **PDM**, sem agregar MDU ou outros tipos. Essa escolha inicial pode ser alterada na célula de configuração.

A leitura das TAGs também reconhece o formato de MBI sem separador entre número e evento, como `PSD_P_MBI1_PDM10FC`. Assim, esses registros passam pela mesma identificação dos demais alarmes.

O período exploratório vai de **17/10/2023 a 28/03/2026**. O início acompanha a retomada de registros em várias estações após meses vazios em 2023. O fim evita avançar além da última data de início de avaria registrada em ZM02, a mais antiga entre os limites finais dos quatro campos de referência. Essas datas **não comprovam cobertura completa**: servem apenas para construir e examinar a estrutura da tabela.

O resumo por estação ajuda a visualizar diferenças entre os períodos registrados. Primeiro e último alarme não representam início e fim de operação.

In [15]:
# Configuração simples do recorte exploratório.
TIPO_ATIVO = "PDM"
DATA_INICIO = pd.Timestamp("2023-10-17")
DATA_FIM = pd.Timestamp("2026-03-28")

assert DATA_INICIO <= DATA_FIM, "O início precisa ser anterior ou igual ao fim."

alarmes_pdm = alarmes.loc[alarmes["tipo_ativo"].eq(TIPO_ATIVO)].copy()

periodo_por_estacao = (
    alarmes_pdm.groupby("estacao")["data_alarme"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "primeiro_registro", "max": "ultimo_registro",
                     "count": "qtd_registros"})
)

display(periodo_por_estacao)


,primeiro_registro,ultimo_registro,qtd_registros
estacao,,,
BUT,2023-10-17,2026-05-12,84434
FAL,2023-10-17,2026-05-12,25011
FRA,2023-10-18,2026-04-14,145188
FRE,2023-02-27,2026-05-12,2111
HIG,2023-10-17,2026-05-12,85184
LUZ,2023-10-17,2026-05-12,39148
MBI,2024-06-05,2025-12-31,72913
PIN,2023-10-17,2025-12-31,114359
PTA,2023-10-24,2025-01-26,29048


## 12. Lista das portas presentes no recorte

A lista reúne identificações completas de PDM encontradas nos alarmes, na ZM01 ou na ZM02 durante o período escolhido. A inclusão das corretivas evita limitar a lista apenas às portas com alarmes.

- `.between()` seleciona datas entre os dois limites, incluindo ambos.
- `pd.concat()` reúne as listas.
- `.drop_duplicates()` mantém uma linha por identificação.

Esta é uma **lista exploratória de ativos registrados**, não um cadastro de portas em operação. O calendário pode conter datas anteriores à primeira identificação de uma porta. A definição de amostras de treino exigirá validar o cadastro e a cobertura, sem usar uma identificação futura como prova de operação no passado. A ZM03 permanece fora desta lista por não identificar portas individualmente.

In [16]:
colunas_porta = ["chave_candidata", "estacao", "plataforma", "porta", "tipo_ativo"]

portas_alarmes = alarmes_pdm.loc[
    alarmes_pdm["data_alarme"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm01 = zm01.loc[
    zm01["tipo_ativo"].eq(TIPO_ATIVO)
    & zm01["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm02 = zm02.loc[
    zm02["tipo_ativo"].eq(TIPO_ATIVO)
    & zm02["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas = pd.concat([portas_alarmes, portas_zm01, portas_zm02], ignore_index=True)
portas = portas.dropna(subset=colunas_porta)
portas = portas.drop_duplicates().sort_values("chave_candidata").reset_index(drop=True)

assert not portas.empty, "Nenhuma porta foi encontrada no recorte."
assert not portas["chave_candidata"].duplicated().any(), "Há identificações conflitantes."

print("Quantidade de portas na lista:", len(portas))
display(portas.head(10))


Quantidade de portas na lista: 528


,chave_candidata,estacao,plataforma,porta,tipo_ativo
0,BUT|1|10|PDM,BUT,1,10,PDM
1,BUT|1|11|PDM,BUT,1,11,PDM
2,BUT|1|12|PDM,BUT,1,12,PDM
3,BUT|1|13|PDM,BUT,1,13,PDM
4,BUT|1|14|PDM,BUT,1,14,PDM
5,BUT|1|15|PDM,BUT,1,15,PDM
6,BUT|1|16|PDM,BUT,1,16,PDM
7,BUT|1|17|PDM,BUT,1,17,PDM
8,BUT|1|18|PDM,BUT,1,18,PDM
9,BUT|1|19|PDM,BUT,1,19,PDM


## 13. Calendário: uma linha por porta e dia

`pd.date_range()` cria todos os dias do período. O `merge(how="cross")` combina cada porta com cada dia.

Por exemplo, uma lista com 2 portas e um calendário de 3 dias produz 6 linhas. A tabela ainda é uma estrutura exploratória; a existência de uma linha não significa que houve coleta de dados naquele dia.

In [17]:
calendario = pd.DataFrame({
    "data": pd.date_range(DATA_INICIO, DATA_FIM, freq="D")
})

grade_diaria = portas.merge(calendario, how="cross")

assert len(grade_diaria) == len(portas) * len(calendario)
assert not grade_diaria.duplicated(["chave_candidata", "data"]).any()

print("Portas:", len(portas))
print("Dias:", len(calendario))
print("Linhas do calendário:", len(grade_diaria))
display(grade_diaria.head())


Portas: 528
Dias: 894
Linhas do calendário: 472032


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21


## 14. Inclusão da contagem diária de alarmes

O `merge(how="left")` mantém todas as linhas do calendário e procura a contagem correspondente à mesma chave e data. `validate="one_to_one"` verifica se cada combinação aparece no máximo uma vez em cada tabela.

A coluna `qtd_registros_alarmes` mantém valores ausentes (`<NA>`) quando não há correspondência. O preenchimento com zero depende de confirmar que a coleta estava completa. A coluna `tem_registro_alarme` indica apenas a presença de registros; `False` não comprova ausência de alarmes.

As contagens ainda incluem possíveis repetições da fonte. Esta versão não está pronta para treinamento.

In [18]:
contagens_recorte = alarmes_diarios.loc[
    alarmes_diarios["chave_candidata"].isin(portas["chave_candidata"])
    & alarmes_diarios["data_alarme"].between(DATA_INICIO, DATA_FIM)
].copy()

contagens_recorte = contagens_recorte.rename(columns={"data_alarme": "data"})

tabela_mestra_inicial = grade_diaria.merge(
    contagens_recorte,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

# Int64, com I maiúsculo, permite números inteiros e valores ausentes.
tabela_mestra_inicial["qtd_registros_alarmes"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].astype("Int64")
)
tabela_mestra_inicial["tem_registro_alarme"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].notna()
)

display(tabela_mestra_inicial.head(10))


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17,<NA>,False
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18,<NA>,False
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19,<NA>,False
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20,<NA>,False
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21,<NA>,False
5,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-22,<NA>,False
6,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-23,<NA>,False
7,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-24,<NA>,False
8,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-25,<NA>,False
9,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-26,<NA>,False


## 15. Conferência da tabela inicial

As verificações conferem o tamanho do calendário, a unicidade de porta/dia e a conservação das contagens. O exemplo exibe dias de uma porta com registros no recorte, incluindo dias sem correspondência quando existirem.

In [19]:
assert len(tabela_mestra_inicial) == len(grade_diaria)
assert not tabela_mestra_inicial.duplicated(["chave_candidata", "data"]).any()
assert (
    tabela_mestra_inicial["qtd_registros_alarmes"].sum()
    == contagens_recorte["qtd_registros_alarmes"].sum()
)

print("Linhas da tabela inicial:", len(tabela_mestra_inicial))
print("Linhas com registros de alarmes:",
      tabela_mestra_inicial["tem_registro_alarme"].sum())
print("Linhas sem correspondência nos alarmes:",
      tabela_mestra_inicial["qtd_registros_alarmes"].isna().sum())

if not contagens_recorte.empty:
    primeiro_registro = contagens_recorte.iloc[0]
    exemplo = tabela_mestra_inicial.loc[
        tabela_mestra_inicial["chave_candidata"].eq(primeiro_registro["chave_candidata"])
        & tabela_mestra_inicial["data"].between(
            primeiro_registro["data"] - pd.Timedelta(days=3),
            primeiro_registro["data"] + pd.Timedelta(days=6),
        )
    ]
    display(exemplo)


Linhas da tabela inicial: 472032
Linhas com registros de alarmes: 47518
Linhas sem correspondência nos alarmes: 424514


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
21,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-07,<NA>,False
22,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-08,<NA>,False
23,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-09,<NA>,False
24,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-10,3,True
25,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-11,3,True
26,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-12,1,True
27,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-13,10,True
28,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-14,<NA>,False
29,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-15,<NA>,False
30,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-16,<NA>,False


### Resultado e próxima etapa

A variável `tabela_mestra_inicial` contém uma linha por PDM e dia, com identificação e contagem de registros de alarmes. O resultado permanece em memória; nenhum CSV é exportado.

A próxima etapa deverá definir a contagem de registros de manutenção, tratando ordens repetidas e o nível de identificação de cada ocorrência. ZM01/ZM02 com identificação completa poderão ser associadas à mesma porta. Registros coletivos e ZM03 precisarão de regras próprias de contexto. A formação do alvo exigirá também a revisão dos registros ZM01 excluídos na limpeza e da cobertura das fontes.

O calendário inicial não confirma operação diária, coleta completa ou ausência de falhas. Essas condições precisam ser resolvidas antes do preenchimento de zeros, das janelas de 30 dias e do alvo de sete dias.

## 16. Regra de contagem das corretivas

Esta etapa acrescenta informações de ZM01 e ZM02 ao calendário de PDM. A unidade contada é a **ordem por porta e dia de início da avaria**. Essa contagem ainda não representa episódios de falha nem manutenções executadas.

Uma ordem repetida com o mesmo local e dia entra apenas uma vez na contagem. Quando a mesma ordem apresenta locais ou dias diferentes, seus registros ficam separados para revisão. Registros sem número de ordem também ficam separados; a nota é preservada na tabela de origem.

A identificação de divergências utiliza a base disponível inteira apenas para auditoria. Ela não representa o conhecimento que existia em cada dia histórico. Antes da modelagem, a versão e a disponibilidade temporal dos registros precisarão ser confirmadas.

O laço `for` aplica a mesma conferência às duas tabelas. `nunique(dropna=False)` conta valores distintos, incluindo ausências, para não esconder divergências.

In [20]:
corretivas = {"ZM01": zm01, "ZM02": zm02}
corretivas_para_revisar = {}
corretivas_sem_ambiguidade = {}
resumo_ordens = []

for nome, tabela in corretivas.items():
    variacoes = tabela.groupby("ordem")[["local_original", "data_avaria"]].nunique(dropna=False)
    ordens_divergentes = variacoes.index[
        (variacoes["local_original"] > 1) | (variacoes["data_avaria"] > 1)
    ]

    precisa_revisao = tabela["ordem"].isna() | tabela["ordem"].isin(ordens_divergentes)
    corretivas_para_revisar[nome] = tabela.loc[precisa_revisao].copy()
    corretivas_sem_ambiguidade[nome] = tabela.loc[~precisa_revisao].copy()

    assert len(corretivas_para_revisar[nome]) + len(corretivas_sem_ambiguidade[nome]) == len(tabela)

    resumo_ordens.append({
        "fonte": nome,
        "linhas_originais": len(tabela),
        "linhas_sem_ordem": tabela["ordem"].isna().sum(),
        "ordens_com_local_ou_dia_divergente": len(ordens_divergentes),
        "linhas_para_revisar": precisa_revisao.sum(),
    })

display(pd.DataFrame(resumo_ordens))
display(corretivas_para_revisar["ZM01"][[
    "ordem", "nota", "local_original", "data_avaria",
]].head(10))


,fonte,linhas_originais,linhas_sem_ordem,ordens_com_local_ou_dia_divergente,linhas_para_revisar
0,ZM01,10543,1,0,1
1,ZM02,1163,5,0,5


,ordem,nota,local_original,data_avaria
5263,NaN,14445462,SP04-6-PSD-MBI-PLT1-PP11PDM,2025-02-07


## 17. Contagens de ZM01 e ZM02 por porta e dia

A seleção considera identificações presentes na lista `portas` e datas dentro do recorte. Os demais tipos, registros coletivos e datas externas permanecem nas tabelas anteriores.

`.drop_duplicates()` gera uma referência por ordem/local/dia **somente para a contagem**. A operação não escolhe qual versão de texto, status ou diagnóstico da ordem está correta.

As colunas finais mantêm o termo `inicio_avaria` para distinguir essa referência de uma data de execução.

In [21]:
corretivas_no_recorte = {}
contagens_corretivas = {}
resumo_corretivas = []

for nome, tabela in corretivas_sem_ambiguidade.items():
    dentro_do_recorte = (
        tabela["chave_candidata"].isin(portas["chave_candidata"])
        & tabela["data_avaria"].between(DATA_INICIO, DATA_FIM)
    )
    selecionadas = tabela.loc[dentro_do_recorte].copy()
    ordens_unicas = selecionadas.drop_duplicates(["ordem", "local_original", "data_avaria"])
    corretivas_no_recorte[nome] = ordens_unicas

    contagem = (
        ordens_unicas.groupby(["chave_candidata", "data_avaria"])
        .size()
        .reset_index(name="qtd_ordens")
        .rename(columns={"data_avaria": "data"})
    )
    contagens_corretivas[nome] = contagem

    assert contagem["qtd_ordens"].sum() == len(ordens_unicas)
    resumo_corretivas.append({
        "fonte": nome,
        "linhas_selecionadas": len(selecionadas),
        "repeticoes_na_contagem": len(selecionadas) - len(ordens_unicas),
        "ordens_contadas": len(ordens_unicas),
    })

zm01_diaria = contagens_corretivas["ZM01"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm01_inicio_avaria",
})
zm02_diaria = contagens_corretivas["ZM02"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm02_inicio_avaria",
})

display(pd.DataFrame(resumo_corretivas))
display(zm01_diaria.head())
display(zm02_diaria.head())


,fonte,linhas_selecionadas,repeticoes_na_contagem,ordens_contadas
0,ZM01,7256,42,7214
1,ZM02,583,1,582


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria
0,BUT|1|10|PDM,2023-11-10,2
1,BUT|1|10|PDM,2023-11-12,1
2,BUT|1|10|PDM,2023-11-13,2
3,BUT|1|10|PDM,2023-12-09,1
4,BUT|1|10|PDM,2023-12-20,1


,chave_candidata,data,qtd_ordens_zm02_inicio_avaria
0,BUT|1|10|PDM,2023-12-09,1
1,BUT|1|15|PDM,2025-08-07,1
2,BUT|1|18|PDM,2024-07-08,1
3,BUT|1|19|PDM,2024-12-04,1
4,BUT|1|23|PDM,2024-09-21,1


## 18. Junção das corretivas ao calendário

Cada junção usa a mesma porta e data. O `how="left"` preserva todas as linhas do calendário. O `validate="one_to_one"` impede a multiplicação de linhas caso uma contagem tenha chaves repetidas.

Valores sem correspondência permanecem ausentes. Em particular, a presença de ordens separadas para revisão e a cobertura ainda não confirmada impedem interpretar essas ausências automaticamente como zero falhas.

In [22]:
tabela_mestra_manutencoes = tabela_mestra_inicial.merge(
    zm01_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm02_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

colunas_corretivas = [
    "qtd_ordens_zm01_inicio_avaria",
    "qtd_ordens_zm02_inicio_avaria",
]
tabela_mestra_manutencoes[colunas_corretivas] = (
    tabela_mestra_manutencoes[colunas_corretivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert tabela_mestra_manutencoes["qtd_ordens_zm01_inicio_avaria"].sum() == zm01_diaria["qtd_ordens_zm01_inicio_avaria"].sum()
assert tabela_mestra_manutencoes["qtd_ordens_zm02_inicio_avaria"].sum() == zm02_diaria["qtd_ordens_zm02_inicio_avaria"].sum()

display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_corretivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_corretivas],
].head(10))


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria,qtd_ordens_zm02_inicio_avaria
24,BUT|1|10|PDM,2023-11-10,2,<NA>
26,BUT|1|10|PDM,2023-11-12,1,<NA>
27,BUT|1|10|PDM,2023-11-13,2,<NA>
53,BUT|1|10|PDM,2023-12-09,1,1
64,BUT|1|10|PDM,2023-12-20,1,<NA>
304,BUT|1|10|PDM,2024-08-16,1,<NA>
323,BUT|1|10|PDM,2024-09-04,2,<NA>
371,BUT|1|10|PDM,2024-10-22,1,<NA>
407,BUT|1|10|PDM,2024-11-27,1,<NA>
494,BUT|1|10|PDM,2025-02-22,1,<NA>


## 19. ZM03: separação por estação e plataforma

A programação preventiva será tratada como **contexto coletivo**, sem indicar uma intervenção comprovada em cada porta.

A seleção reconhece somente os locais completos nos formatos:

- `SP04-6-PSD-BUT`: contexto da estação.
- `SP04-6-PSD-BUT-PLT1`: contexto da plataforma.

Um local genérico como `SP04-6-PSD` fica separado. A correspondência exata evita tratar um subcomponente como manutenção de toda a estação.

Nesta base, cada ordem ZM03 aparece uma vez. A verificação interrompe a contagem se uma versão futura contiver ordens repetidas, ausentes ou status de cancelamento desconhecidos. A coluna `cancelada` descreve o cadastro disponível: seu valor não comprova o status conhecido na data programada.

In [23]:
assert zm03["ordem"].notna().all(), "Há ordens ZM03 sem identificação."
assert not zm03["ordem"].duplicated().any(), "As ordens ZM03 repetidas precisam de revisão."
assert zm03["cancelada"].isin([True, False]).all(), "Há status de cancelamento desconhecido."

locais_zm03 = zm03["local_original"].astype("string").str.strip().str.upper()

local_estacao = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}", na=False)
local_plataforma = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}-PLT\d+", na=False)
data_no_recorte = zm03["data_programada"].between(DATA_INICIO, DATA_FIM)

zm03_estacao = zm03.loc[local_estacao & data_no_recorte].copy()
zm03_plataforma = zm03.loc[local_plataforma & data_no_recorte].copy()
zm03_sem_escopo = zm03.loc[~(local_estacao | local_plataforma)].copy()

print("Ordens de estação no recorte:", len(zm03_estacao))
print("Ordens de plataforma no recorte:", len(zm03_plataforma))
print("Ordens sem escopo reconhecido na base completa:", len(zm03_sem_escopo))
display(zm03_sem_escopo[["ordem", "local_original", "data_programada"]].head())


Ordens de estação no recorte: 128
Ordens de plataforma no recorte: 230
Ordens sem escopo reconhecido na base completa: 83


,ordem,local_original,data_programada
260,52907929,SP04-6-PSD,2024-07-29
261,52907928,SP04-6-PSD,2024-07-15
262,52907927,SP04-6-PSD,2024-07-08
263,52907926,SP04-6-PSD,2024-07-01
264,52907925,SP04-6-PSD,2024-06-24


## 20. Contagens da programação preventiva

A agregação separa ordens canceladas e não canceladas na versão disponível. **Não cancelada não significa executada.** As quatro colunas são de auditoria da programação e ainda não são entradas aprovadas para o modelo.

Uma mesma ordem de estação não entra também na contagem de plataforma. As duas tabelas preservam os níveis originais.

In [24]:
# Conversão explícita para booleano após a conferência do status.
zm03_estacao["cancelada"] = zm03_estacao["cancelada"].astype(bool)
zm03_estacao["nao_cancelada"] = ~zm03_estacao["cancelada"]

zm03_estacao_diaria = (
    zm03_estacao.groupby(["estacao", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_estacao=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_estacao=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

zm03_plataforma["cancelada"] = zm03_plataforma["cancelada"].astype(bool)
zm03_plataforma["nao_cancelada"] = ~zm03_plataforma["cancelada"]

zm03_plataforma_diaria = (
    zm03_plataforma.groupby(["estacao", "plataforma", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_plataforma=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_plataforma=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

assert zm03_estacao_diaria.filter(like="qtd_").sum().sum() == len(zm03_estacao)
assert zm03_plataforma_diaria.filter(like="qtd_").sum().sum() == len(zm03_plataforma)

display(zm03_estacao_diaria.head())
display(zm03_plataforma_diaria.head())


,estacao,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao
0,BUT,2023-10-30,1,0
1,BUT,2023-11-26,1,0
2,BUT,2023-12-28,1,0
3,BUT,2024-01-24,1,0
4,BUT,2024-02-23,1,0


,estacao,plataforma,data,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
0,BUT,1,2024-01-02,1,0
1,BUT,1,2024-03-18,1,0
2,BUT,1,2024-06-03,1,0
3,BUT,1,2024-09-09,1,0
4,BUT,1,2024-12-05,1,0


## 21. Inclusão do contexto de ZM03

O contexto de estação é associado por estação e data. O de plataforma é associado por estação, plataforma e data. `validate="many_to_one"` permite que várias portas recebam o mesmo contexto, mas exige uma única linha na tabela coletiva para cada chave.

Por exemplo, uma ordem programada para BUT/PLT1 aparece como contexto nas linhas das PDM dessa plataforma naquele dia. Isso não representa uma ordem executada em cada PDM. Por esse motivo, essas colunas **não devem ser somadas entre portas** para obter o total de ordens.

As tabelas agregadas anteriores são a referência para totais de programação. Registros de estações ou plataformas sem PDM na lista continuam disponíveis nessas tabelas.

In [25]:
tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_estacao_diaria,
    on=["estacao", "data"],
    how="left",
    validate="many_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_plataforma_diaria,
    on=["estacao", "plataforma", "data"],
    how="left",
    validate="many_to_one",
)

colunas_preventivas = [
    "qtd_zm03_nao_canceladas_estacao",
    "qtd_zm03_canceladas_estacao",
    "qtd_zm03_nao_canceladas_plataforma",
    "qtd_zm03_canceladas_plataforma",
]
tabela_mestra_manutencoes[colunas_preventivas] = (
    tabela_mestra_manutencoes[colunas_preventivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert not tabela_mestra_manutencoes.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_manutencoes["qtd_registros_alarmes"].equals(
    tabela_mestra_inicial["qtd_registros_alarmes"]
)

print("Linhas após as quatro junções:", len(tabela_mestra_manutencoes))
print("Colunas adicionadas:", len(colunas_corretivas + colunas_preventivas))
display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_preventivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_preventivas],
].head(10))


Linhas após as quatro junções: 472032
Colunas adicionadas: 6


,chave_candidata,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
13,BUT|1|10|PDM,2023-10-30,1,0,<NA>,<NA>
40,BUT|1|10|PDM,2023-11-26,1,0,<NA>,<NA>
72,BUT|1|10|PDM,2023-12-28,1,0,<NA>,<NA>
77,BUT|1|10|PDM,2024-01-02,<NA>,<NA>,1,0
99,BUT|1|10|PDM,2024-01-24,1,0,<NA>,<NA>
129,BUT|1|10|PDM,2024-02-23,1,0,<NA>,<NA>
153,BUT|1|10|PDM,2024-03-18,1,0,1,0
183,BUT|1|10|PDM,2024-04-17,1,0,<NA>,<NA>
230,BUT|1|10|PDM,2024-06-03,<NA>,<NA>,1,0
326,BUT|1|10|PDM,2024-09-07,1,0,<NA>,<NA>


### Resultado da integração de manutenções

A variável `tabela_mestra_manutencoes` reúne o calendário de PDM, as contagens de registros de alarmes, as ordens ZM01/ZM02 por início da avaria e o contexto de programação ZM03. O resultado permanece em memória e os arquivos de origem permanecem intactos.

As tabelas `corretivas_para_revisar`, `zm03_sem_escopo` e a aba `Outliers_removidos` de ZM01 preservam casos que exigem análise antes da definição do alvo. As bases anteriores também mantêm registros coletivos e outros tipos de ativo que não entraram nas contagens individuais.

**Próximo passo:** revisar o critério de falha ZM01, os registros separados e o período de cobertura. Essas decisões permitem distinguir dias sem falha de dias sem informação. Somente depois poderão ser criados `falha_no_dia`, `falha_7d` e as variáveis históricas. Datas programadas de ZM03 e status atuais não serão usados automaticamente como histórico de execução.